# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ihSomik/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

Simple words, honest numbers. All results below are **observed** on one sampled month and are **directional**, for decision-support only.

## 1. Unit of analysis + time window

### The Contract (Plain Words)
1. **What one row means for our lane:** One row is one day of organic performance (`fact_content_daily_performance`) at the grain `report_date` + `client_hash_id` + `content_hash_id` (one published URL / target keyword group for one client on one day). *(Verified by a query in Section 3.)*
2. **Tables we will use:** `fact_content_daily_performance` for performance, joined to `dim_content` for content attributes (word count, search volume, competition).
3. **Time window:** March 2026 (`month=2026-03`), a mid-panel month. The final sample month (`2026-06`) is avoided because its outcome window may be incomplete. *(Section 3 has a query that checks this.)*
4. **Label / proxy:** `has_organic_engagement = (gsc_clicks > 0) OR (ga4_engaged_sessions > 0)` on a given day. This is the same label used in the model in Section 4.
5. **Deliberately excluded:** `sessions_paid` and all paid-channel columns. Paid traffic follows bidding rules, not organic SEO.

In [1]:
import os
import numpy as np
import pandas as pd
from huggingface_hub import login, HfApi

# Token comes from Colab Secrets
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")

if not token:
    raise RuntimeError("HF_TOKEN not found. Add it in Colab -> Secrets (key icon) and enable notebook access.")

os.environ["HF_TOKEN"] = token
login(token=token)
print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [2]:
!pip install -q datasets huggingface_hub pandas pyarrow scikit-learn

In [3]:
api = HfApi()
files = api.list_repo_files(repo_id="FlyRank/internship-warehouse", repo_type="dataset")
print("Data files in FlyRank/internship-warehouse:")
for f in sorted(files):
    if f.endswith((".parquet", ".csv", ".json")):
        print(" -", f)

Data files in FlyRank/internship-warehouse:
 - dim_clients.parquet
 - dim_content.parquet
 - fact_content_daily_performance/month=2025-01/data_0.parquet
 - fact_content_daily_performance/month=2025-02/data_0.parquet
 - fact_content_daily_performance/month=2025-03/data_0.parquet
 - fact_content_daily_performance/month=2025-04/data_0.parquet
 - fact_content_daily_performance/month=2025-05/data_0.parquet
 - fact_content_daily_performance/month=2025-06/data_0.parquet
 - fact_content_daily_performance/month=2025-07/data_0.parquet
 - fact_content_daily_performance/month=2025-08/data_0.parquet
 - fact_content_daily_performance/month=2025-09/data_0.parquet
 - fact_content_daily_performance/month=2025-10/data_0.parquet
 - fact_content_daily_performance/month=2025-11/data_0.parquet
 - fact_content_daily_performance/month=2025-12/data_0.parquet
 - fact_content_daily_performance/month=2026-01/data_0.parquet
 - fact_content_daily_performance/month=2026-02/data_0.parquet
 - fact_content_daily_perfor

In [4]:
storage_options = {"token": token}
BASE = "hf://datasets/FlyRank/internship-warehouse"

df_performance = pd.read_parquet(f"{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet", storage_options=storage_options)
df_content = pd.read_parquet(f"{BASE}/dim_content.parquet", storage_options=storage_options)
df_clients = pd.read_parquet(f"{BASE}/dim_clients.parquet", storage_options=storage_options)

print("fact_content_daily_performance columns:\n", df_performance.columns.tolist())
print("\ndim_content columns:\n", df_content.columns.tolist())
print("\nRows -> performance:", len(df_performance), "| content:", len(df_content), "| clients:", len(df_clients))

fact_content_daily_performance columns:
 ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']

dim_content columns:
 ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimiz

## 2. Fields: feature / label / context / excluded

### Label (or proxy)
- `has_organic_engagement`: `(gsc_clicks > 0) | (ga4_engaged_sessions > 0)`.

### Features (all knowable before the day being predicted)
| Feature | Why it is knowable at the decision moment |
|---|---|
| `char_count` | Content length is fixed when the page is published. |
| `word_count` | Same: part of the written content. |
| `search_volume` | Keyword volume is researched before publishing. |
| `competition_level` | Keyword difficulty is analysed before publishing. |
| `day_of_week` | The calendar is known in advance. |
| `gsc_impressions_lag1d` | Impressions from the **previous day**, so they are already observed when we predict today. |

Missing-value flags (`*_missing`) are added for content fields so "unknown" is not confused with "zero".

### Context (kept for grouping / joins, not used as features)
`report_date`, `client_hash_id`, `content_hash_id`, `month`.

### Excluded (with why)
- `sessions_paid`: driven by ad budgets, not organic SEO.
- `ai_chatgpt`: an outcome measured after the fact, not known at the decision moment.
- Same-day `gsc_clicks`, `gsc_impressions`, `gsc_sum_position`, `ga4_engaged_sessions`: same-day outcomes. They are part of, or too close to, the label.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
print("--- FACT 1: Grain is unique ---")
grain = ["report_date", "client_hash_id", "content_hash_id"]
dups = df_performance.duplicated(subset=grain).sum()
print("Duplicate rows on grain:", dups, "| Grain unique:", dups == 0)

print("\n--- FACT 2: Row count and date span (March 2026) ---")
print("Rows:", len(df_performance))
print("Date span:", df_performance["report_date"].min(), "to", df_performance["report_date"].max())

print("\n--- FACT 3: Availability (GSC flags are TRUE) ---")
mask = (df_performance["client_has_gsc"] == True) & (df_performance["gsc_data_available"] == True)
print(f"Rows with GSC data available: {mask.sum()} of {len(df_performance)} ({mask.mean()*100:.2f}%)")

print("\n--- FACT 4: dim_content has one row per content_hash_id (safe to join) ---")
print("content_hash_id unique in dim_content:", df_content["content_hash_id"].is_unique)

print("\n--- FACT 5: Missing values in the columns we use ---")
use_cols = [c for c in ["gsc_clicks", "gsc_impressions", "ga4_engaged_sessions"] if c in df_performance.columns]
print(df_performance[use_cols].isna().mean().round(4).to_string())
print(df_content[["char_count", "word_count", "search_volume", "competition_level"]].isna().mean().round(4).to_string())

--- FACT 1: Grain is unique ---
Duplicate rows on grain: 0 | Grain unique: True

--- FACT 2: Row count and date span (March 2026) ---
Rows: 9841378
Date span: 2026-03-01 to 2026-03-31

--- FACT 3: Availability (GSC flags are TRUE) ---
Rows with GSC data available: 3611061 of 9841378 (36.69%)

--- FACT 4: dim_content has one row per content_hash_id (safe to join) ---
content_hash_id unique in dim_content: True

--- FACT 5: Missing values in the columns we use ---
gsc_clicks              0.0000
gsc_impressions         0.0000
ga4_engaged_sessions    0.3067
char_count           0.3421
word_count           0.3421
search_volume        0.2745
competition_level    0.2780


In [6]:
# FACT 6: why March and not June? Compare the last sample month's coverage.
try:
    df_june = pd.read_parquet(f"{BASE}/fact_content_daily_performance/month=2026-06/data_0.parquet", storage_options=storage_options)
    print("June rows:", len(df_june))
    print("June date span:", df_june["report_date"].min(), "to", df_june["report_date"].max())
    print("March rows:", len(df_performance))
    # Observed engagement rate per month (directional check for an incomplete window)
    for name, d in [("March", df_performance), ("June", df_june)]:
        lab = ((d["gsc_clicks"].fillna(0) > 0) | (d["ga4_engaged_sessions"].fillna(0) > 0)).mean()
        print(f"{name} observed engagement rate: {lab:.4f}")
    del df_june
except Exception as e:
    print("Could not run the June comparison:", e)

June rows: 11694072
June date span: 2026-06-01 to 2026-06-30
March rows: 9841378
March observed engagement rate: 0.0433
June observed engagement rate: 0.0396


## 4. Building 6 Features and Testing the Leakage Trap

We build the features, then run **the trap**: add one column copied directly from the label, train a classifier, see the fake near-perfect score, delete the column, and compare with the honest score. We split by `content_hash_id` so the same URL never sits in both train and test.

In [7]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 1) Keep only rows where GSC data is available
base = df_performance[(df_performance["client_has_gsc"] == True) & (df_performance["gsc_data_available"] == True)].copy()
base["report_date"] = pd.to_datetime(base["report_date"])

# 2) Label (same definition as the contract)
base["has_organic_engagement"] = (
    (base["gsc_clicks"].fillna(0) > 0) | (base["ga4_engaged_sessions"].fillna(0) > 0)
).astype(int)

# 3) Feature: previous-day impressions, built by date (not row position), on the FULL month before sampling
imp_col = "gsc_impressions"
assert imp_col in base.columns, f"Expected column {imp_col} in fact table"
lag = base[["client_hash_id", "content_hash_id", "report_date", imp_col]].copy()
lag["report_date"] = lag["report_date"] + pd.Timedelta(days=1)   # yesterday's value lands on today's date
lag = lag.rename(columns={imp_col: "gsc_impressions_lag1d"})
base = base.merge(lag, on=["client_hash_id", "content_hash_id", "report_date"], how="left")
# First day of the month has no yesterday inside this file: keep as NaN, flag it, then fill
base["lag1d_missing"] = base["gsc_impressions_lag1d"].isna().astype(int)

# 4) Sample, then merge static content attributes
sample_df = base.sample(n=min(100_000, len(base)), random_state=42).copy()
sample_df = sample_df.merge(
    df_content[["content_hash_id", "char_count", "word_count", "search_volume", "competition_level"]],
    on="content_hash_id", how="left",
)

# 5) Missing values: flag "unknown" instead of pretending it is zero / LOW
for c in ["char_count", "word_count", "search_volume"]:
    sample_df[f"{c}_missing"] = sample_df[c].isna().astype(int)
sample_df["comp_missing"] = sample_df["competition_level"].isna().astype(int)

comp_map = {"LOW": 1, "MEDIUM": 2, "HIGH": 3}
sample_df["feat_competition_level"] = sample_df["competition_level"].str.upper().map(comp_map).fillna(0)  # 0 = unknown

sample_df["feat_char_count"] = sample_df["char_count"].fillna(0)
sample_df["feat_word_count"] = sample_df["word_count"].fillna(0)
sample_df["feat_search_volume"] = sample_df["search_volume"].fillna(0)
sample_df["feat_day_of_week"] = sample_df["report_date"].dt.dayofweek
sample_df["feat_impressions_lag1d"] = sample_df["gsc_impressions_lag1d"].fillna(0)

honest_features = [
    "feat_char_count", "feat_word_count", "feat_search_volume",
    "feat_competition_level", "feat_day_of_week", "feat_impressions_lag1d",
    "char_count_missing", "word_count_missing", "search_volume_missing",
    "comp_missing", "lag1d_missing",
]
y = sample_df["has_organic_engagement"]
groups = sample_df["content_hash_id"]
print("Sample rows:", len(sample_df))
print("Label balance (observed):", y.value_counts(normalize=True).round(4).to_dict())

Sample rows: 100000
Label balance (observed): {0: 0.8829, 1: 0.1171}


In [8]:
# Group-aware split: the same content_hash_id never appears in both train and test
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(sample_df, y, groups))
train, test = sample_df.iloc[train_idx], sample_df.iloc[test_idx]
print("Shared URLs between train and test:", len(set(train["content_hash_id"]) & set(test["content_hash_id"])))

def fit_and_score(features, label="model"):
    clf = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42, n_jobs=-1)
    clf.fit(train[features], train["has_organic_engagement"])
    auc = roc_auc_score(test["has_organic_engagement"], clf.predict_proba(test[features])[:, 1])
    print(f"{label} AUC (observed): {auc:.5f}")
    return auc

# THE TRAP: a column derived directly from the label
sample_df["trap_leaked_feature"] = sample_df["has_organic_engagement"]
train, test = sample_df.iloc[train_idx], sample_df.iloc[test_idx]
leaked_auc = fit_and_score(honest_features + ["trap_leaked_feature"], "LEAKED")

# DELETE the leaked column, then score honestly
sample_df = sample_df.drop(columns=["trap_leaked_feature"])
train, test = sample_df.iloc[train_idx], sample_df.iloc[test_idx]
print("Leaked column removed:", "trap_leaked_feature" not in sample_df.columns)
honest_auc = fit_and_score(honest_features, "HONEST")

print(f"\nAUC drop after removing the leak: {leaked_auc - honest_auc:.5f}")
print("Reading (directional): a near-perfect score from a label copy is a warning sign, not a good model.")

Shared URLs between train and test: 0
LEAKED AUC (observed): 1.00000
Leaked column removed: True
HONEST AUC (observed): 0.84987

AUC drop after removing the leak: 0.15013
Reading (directional): a near-perfect score from a label copy is a warning sign, not a good model.


### Other leak candidates we checked and did not use
`gsc_sum_position` (same-day position) is only filled when the page had impressions that day. A missing or zero position therefore hints that there were no clicks. This is a **leak by availability**, so it stays out of the honest feature set. Same-day `gsc_impressions` is excluded for the same reason; only the previous-day value (`gsc_impressions_lag1d`) is used.

## 5. Data limits

*What can this data never tell you?*

### What this data can never tell you (Named Limitations)
1. **No long-run conversion trail:** The table is top-of-funnel traffic. It cannot trace multi-touch conversions or offline outcomes that happen weeks later.
2. **Off-page changes are invisible:** Features are on-page only. Search algorithm updates and competitor shifts (e.g. backlink growth) are not measured.
3. **Search intent shifts:** Keyword intent labels are static; they do not show seasonal changes in intent.
4. **One month, one sample:** Results use March 2026 and a 100k-row sample, so they are observed and directional, not proof of a general rule.
5. **GSC-only rows:** Only rows with GSC data available are kept, so clients without it are not represented.
6. **Lag feature edge:** The first day of the month has no previous day in this file, so its lag value is flagged missing.

## Self-check

- [x] Every section is filled: markdown thinking AND code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all) *(run it once before submitting)*
- [x] No client names, URLs, tokens, or private queries in the notebook
- [x] Claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/`, then submit the repo URL on the card